In [3]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

# -------------------------
# 1) Model components
# -------------------------
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        num_groups = min(8, out_ch)
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(num_groups, out_ch),
            nn.SiLU()
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(num_groups, out_ch),
            nn.SiLU()
        )
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t):
        h = self.block1(x)
        time_emb = self.time_mlp(t)[:, :, None, None]
        h = h + time_emb
        h = self.block2(h)
        return h + self.res_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, channel_mults=(1,2,4,8), time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim)
        )
        self.downs = nn.ModuleList()
        in_chs = [in_channels] + [base_channels * m for m in channel_mults]
        for i in range(len(channel_mults)):
            self.downs.append(ResidualBlock(in_chs[i], in_chs[i+1], time_dim))
            self.downs.append(nn.Conv2d(in_chs[i+1], in_chs[i+1], 3, stride=2, padding=1))
        self.bottleneck = ResidualBlock(in_chs[-1], in_chs[-1], time_dim)
        self.ups = nn.ModuleList()
        for i in reversed(range(len(channel_mults))):
            self.ups.append(nn.ConvTranspose2d(in_chs[i+1], in_chs[i+1], 4, stride=2, padding=1))
            self.ups.append(ResidualBlock(in_chs[i+1]*2, in_chs[i], time_dim))
        self.final = nn.Conv2d(base_channels, in_channels, 1)

    def forward(self, x, t):
        t = self.time_mlp(t)
        skips = []
        for layer in self.downs:
            if isinstance(layer, ResidualBlock):
                x = layer(x, t)
                skips.append(x)
            else:
                x = layer(x)
        x = self.bottleneck(x, t)
        for layer in self.ups:
            if isinstance(layer, nn.ConvTranspose2d):
                x = layer(x)
            else:
                skip = skips.pop()
                x = torch.cat([x, skip], dim=1)
                x = layer(x, t)
        return self.final(x)

# -------------------------
# 2) Diffusion schedule
# -------------------------
beta_start, beta_end, T = 1e-4, 0.02, 1000
betas = torch.linspace(beta_start, beta_end, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

@torch.no_grad()
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    tb = t.cpu()
    sqrt_alpha_bar = alpha_bars[tb].sqrt().view(-1,1,1,1).to(x0.device)
    sqrt_one_minus_alpha_bar = (1 - alpha_bars[tb]).sqrt().view(-1,1,1,1).to(x0.device)
    return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise, noise

@torch.no_grad()
def p_sample(model, x, t):
    tb = t.cpu()
    betas_t = betas[tb].view(-1,1,1,1).to(x.device)
    sqrt_one_minus_alpha_bar = (1 - alpha_bars[tb]).sqrt().view(-1,1,1,1).to(x.device)
    sqrt_recip_alphas = (1.0 / alphas[tb].sqrt()).view(-1,1,1,1).to(x.device)
    eps_pred = model(x, t.float())
    mean = sqrt_recip_alphas * (x - betas_t / sqrt_one_minus_alpha_bar * eps_pred)
    if t[0] > 0:
        noise = torch.randn_like(x)
        sigma = betas_t.sqrt()
        return mean + sigma * noise
    else:
        return mean

@torch.no_grad()
def sample_images(model, n_samples, device):
    model.eval()
    x = torch.randn(n_samples, 3, 32, 32, device=device)
    for i in reversed(range(T)):
        t = torch.full((n_samples,), i, device=device, dtype=torch.long)
        x = p_sample(model, x, t)
    return x.clamp(-1,1)

# -------------------------
# 3) Training utilities
# -------------------------
def train_diffusion(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for x, _ in dataloader:
        x = x.to(device)
        bsz = x.size(0)
        t = torch.randint(0, T, (bsz,), device=device)
        x_noisy, noise = q_sample(x, t)
        noise_pred = model(x_noisy, t.float())
        loss = F.mse_loss(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def eval_diffusion(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            bsz = x.size(0)
            t = torch.randint(0, T, (bsz,), device=device)
            x_noisy, noise = q_sample(x, t)
            noise_pred = model(x_noisy, t.float())
            loss = F.mse_loss(noise_pred, noise)
            total_loss += loss.item()
    return total_loss / len(dataloader)

# -------------------------
# 4) Main script
# -------------------------
if __name__ == "__main__":
    epochs = 50
    batch_size = 128
    lr = 1e-4
    patience = 5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    n_train = int(len(dataset) * 0.8)
    n_val = len(dataset) - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val = float('inf')
    epochs_no_improve = 0
    train_losses, val_losses = [], []

    for epoch in range(1, epochs+1):
        train_loss = train_diffusion(model, train_loader, optimizer, device)
        val_loss = eval_diffusion(model, val_loader, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_ddpm.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break

    model.load_state_dict(torch.load("best_ddpm.pth"))

    plt.figure(figsize=(8,4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.show()

    samples = sample_images(model, n_samples=16, device=device)
    grid = make_grid((samples * 0.5 + 0.5), nrow=4)
    plt.figure(figsize=(6,6))
    plt.axis('off')
    plt.imshow(grid.permute(1,2,0).cpu())
    plt.show()


Files already downloaded and verified
Files already downloaded and verified


RuntimeError: Given groups=1, weight of size [3, 64, 1, 1], expected input[128, 3, 32, 32] to have 64 channels, but got 3 channels instead